In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
# ============================================================
# 0. INSTALLATION
# ============================================================

# Run this once in Colab/Jupyter if required.
# !pip install -U transformers datasets peft accelerate


# ============================================================
# 1. IMPORTS
# ============================================================

import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model
)


# ============================================================
# 2. CHECK GPU
# ============================================================

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 3. MODEL NAME
# ============================================================

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"


# ============================================================
# 4. LOAD TOKENIZER
# ============================================================

# Tokenizer converts:
#
# Text
#   ↓
# Tokens
#   ↓
# Token IDs
#
# Example:
# "Hello world"
#       ↓
# [token1, token2]
#       ↓
# [15496, 995]

tokenizer = AutoTokenizer.from_pretrained(model_name)


# TinyLlama may not have a padding token.
# For causal LM training, we can use EOS as padding.

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ============================================================
# 5. CREATE / LOAD DATASET
# ============================================================

# ------------------------------------------------------------
# Example dataset
# ------------------------------------------------------------
# Replace this section with your actual dataset.
#
# Your dataset should ultimately contain a column containing
# the training text.
# ------------------------------------------------------------

texts = [
    "Machine learning is a field of artificial intelligence.",
    "Deep learning uses neural networks to learn representations.",
    "Transformers use attention mechanisms to process sequences.",
    "LoRA is a parameter efficient fine tuning technique.",
    "Large language models can be fine tuned using LoRA."
]

dataset = Dataset.from_dict({
    "text": texts
})

print(dataset)


# ============================================================
# 6. TOKENIZE DATASET
# ============================================================

# Maximum sequence length.
#
# Increase this if your GPU has enough memory.
# 256 / 512 is good for a small experiment.

MAX_LENGTH = 256


def tokenize_function(example):

    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print(tokenized_dataset)


# ============================================================
# 7. DATA COLLATOR
# ============================================================

# For causal language modeling:
#
# Input:
#     I love machine learning
#
# Model learns:
#     I → love
#     I love → machine
#     I love machine → learning
#
# mlm=False means:
# "Do NOT use masked language modeling."
#
# Instead, train using next-token prediction.

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)


# ============================================================
# 8. LOAD PRETRAINED MODEL
# ============================================================

# IMPORTANT:
#
# We are NOT using:
#
#     load_in_8bit=True
#
# here.
#
# We are loading the model in FP16 to avoid the
# bitsandbytes CUDA issue you encountered.

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)


# ============================================================
# 9. DISABLE CACHE DURING TRAINING
# ============================================================

# Caching is useful during generation,
# but it is normally disabled during training.

model.config.use_cache = False


# ============================================================
# 10. DEFINE LoRA CONFIGURATION
# ============================================================

lora_config = LoraConfig(

    # We are doing causal language modeling.
    task_type=TaskType.CAUSAL_LM,

    # Rank of LoRA matrices.
    #
    # Higher r:
    #     More trainable parameters
    #     More capacity
    #
    # Lower r:
    #     Fewer parameters
    #     Less memory

    r=8,

    # Scaling factor for LoRA updates.

    lora_alpha=16,

    # Dropout applied to LoRA layers.

    lora_dropout=0.05,

    # We don't train bias parameters.

    bias="none",

    # LoRA will be inserted into these
    # attention projection layers.

    target_modules=[
        "q_proj",
        "v_proj"
    ]
)


# ============================================================
# 11. ATTACH LoRA TO THE MODEL
# ============================================================

# Original model:
#
#     TinyLlama
#        ↓
#     1.1B parameters
#
# After get_peft_model():
#
#     TinyLlama
#        +
#     LoRA adapters
#
# During training, the original model parameters
# remain frozen and only LoRA parameters are trained.

model = get_peft_model(
    model,
    lora_config
)


# ============================================================
# 12. CHECK TRAINABLE PARAMETERS
# ============================================================

model.print_trainable_parameters()

# You should see something similar to:
#
# trainable params: ... || all params: ... ||
# trainable%: ...
#
# The trainable percentage should be very small.


# ============================================================
# 13. TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(

    # Where the LoRA adapter/checkpoints will be saved.

    output_dir="./tinyllama-lora",

    # Number of complete passes through the dataset.

    num_train_epochs=5,

    # Number of examples processed simultaneously
    # by one GPU.

    per_device_train_batch_size=1,

    # Simulate a larger batch size.
    #
    # Effective batch size =
    #
    # batch_size × gradient_accumulation_steps
    #
    # = 1 × 8
    # = 8

    gradient_accumulation_steps=8,

    # Learning rate for LoRA parameters.

    learning_rate=2e-4,

    # Use FP16 training.

    fp16=True,

    # Print training information every 20 steps.

    logging_steps=20,

    # Keep only one checkpoint.

    save_total_limit=1,

    # Don't send logs to WandB/etc.

    report_to="none",

    # Save checkpoint at the end of each epoch.

    save_strategy="epoch"
)


# ============================================================
# 14. CREATE TRAINER
# ============================================================

trainer = Trainer(

    # LoRA-enabled model

    model=model,

    # Training configuration

    args=training_args,

    # Tokenized training data

    train_dataset=tokenized_dataset,

    # Creates labels and handles padding.

    data_collator=data_collator
)


# ============================================================
# 15. START TRAINING
# ============================================================

trainer.train()


# ============================================================
# 16. SAVE LoRA ADAPTER
# ============================================================

# IMPORTANT:
#
# This does NOT save a completely new 1.1B model.
#
# It saves only the small LoRA adapter.

adapter_path = "./tinyllama-lora-final"

model.save_pretrained(adapter_path)

tokenizer.save_pretrained(adapter_path)

print("LoRA adapter saved to:", adapter_path)

CUDA available: True
GPU: Tesla T4
Dataset({
    features: ['text'],
    num_rows: 5
})


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 5
})


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


Step,Training Loss


[transformers] mtime may not be reliable on this filesystem, falling back to numerical ordering


LoRA adapter saved to: ./tinyllama-lora-final


In [4]:
# ============================================================
# LOAD BASE MODEL + LoRA ADAPTER
# ============================================================

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# Load the original/base TinyLlama
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Load your trained LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    "./tinyllama-lora-final"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "./tinyllama-lora-final"
)

# Put model in evaluation mode
model.eval()


# ============================================================
# INTERACTIVE QUESTION-ANSWERING
# ============================================================

while True:

    # Ask the user for a question
    question = input("\nYou: ")

    # Stop the loop
    if question.lower() in ["exit", "quit", "q"]:
        print("Exiting...")
        break

    # Convert question → token IDs
    inputs = tokenizer(
        question,
        return_tensors="pt"
    ).to(model.device)

    # Generate answer
    with torch.no_grad():

        outputs = model.generate(
            **inputs,

            # Number of tokens to generate
            max_new_tokens=100,

            # Enable sampling
            do_sample=True,

            # Controls randomness
            temperature=0.7,

            # Avoid excessive repetition
            repetition_penalty=1.1,

            # Stop when EOS token is generated
            eos_token_id=tokenizer.eos_token_id
        )

    # Convert token IDs → text
    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("\nTinyLlama:", answer)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


You:  what is llm


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TinyLlama: what is llm, lls, and lla
#
# ## What is the difference between these?
# The first one is for locating a location of a file in your machine, the second one is for finding the path to a file.
#
# ### What are the differences between dir() and dirname() ?
#
# It returns the directory names at which the file or directory can be found.
#
# If we want only files/directories with certain name



You:  oRA


...


TinyLlama: lora
# 40:17720_0069_0100_00103_00000_y_lora
# 40:17720_0069_0100_0070_00000_y_lora
# 40:17720_0069_0100_008



You:  what is lora


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TinyLlama: what is lora, and the history behind it.


KeyboardInterrupt: Interrupted by user

In [2]:
!pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 37.3 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
